# OULAD analysis

## Prerequisite
- pip install packages needed below
- Download dataset from OULAD

In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np

In [2]:
base_data_file_name = 'data_base.csv'
data_file_name = 'data.csv'
class_num = 3
weighted = False  # whether consider the weight of quizz

# Download data from OULAD first
studentInfo = pd.read_csv('studentInfo.csv')
studentAssess = pd.read_csv('studentAssessment.csv')
assessments = pd.read_csv('assessments.csv')
studentRegistration = pd.read_csv('studentRegistration.csv')

# keep only 'assessment_type' == 'Exam' in assessments
assessments = assessments[assessments['assessment_type'] == 'Exam']

merged_df = pd.merge(assessments, studentAssess, on=['id_assessment'], how='inner')
merged_df = pd.merge(merged_df, studentInfo, on=['code_module', 'code_presentation', 'id_student'], how='left')
merged_df = pd.merge(merged_df, studentRegistration, on=['code_module', 'code_presentation', 'id_student'], how='left')

print(merged_df.shape)
merged_df.head()

(4959, 21)


,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score,...,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration
0,CCC,2014B,24290,Exam,NaN,100.0,558914,230,0,32.0,...,North Western Region,A Level or Equivalent,10-20,0-35,0,90,N,Fail,-74.0,NaN
1,CCC,2014B,24290,Exam,NaN,100.0,559706,234,0,78.0,...,East Midlands Region,Lower Than A Level,60-70%,0-35,0,30,N,Pass,-22.0,NaN
2,CCC,2014B,24290,Exam,NaN,100.0,559770,230,0,54.0,...,South East Region,HE Qualification,90-100%,35-55,0,60,N,Pass,-22.0,NaN
3,CCC,2014B,24290,Exam,NaN,100.0,560114,230,0,64.0,...,North Western Region,A Level or Equivalent,10-20,0-35,0,30,Y,Pass,-281.0,NaN
4,CCC,2014B,24290,Exam,NaN,100.0,560311,234,0,100.0,...,East Anglian Region,Lower Than A Level,50-60%,0-35,0,60,N,Distinction,-28.0,NaN


# Normalization

In [3]:
def normalize(df):
    df = df.copy()
    # df = pd.read_csv('dataMerged.csv')

    # 找出非 nominal 欄位（數值欄位，但排除 nominal 和 target 欄位，如果你有一個叫 'score' 的 target 欄位）
    non_nominal_cols = df.select_dtypes(include=['number']).columns.difference(['score', 'avg_score', 'late_ratio', 'completion_ratio', 'id_student', 'score_diff'])

    # 初始化 scaler
    scaler = MinMaxScaler()

    # 對非 nominal 欄位做 normalization，並直接覆蓋原欄位
    df[non_nominal_cols] = scaler.fit_transform(df[non_nominal_cols])

    print(df.describe())
    return df

# Assessments Stats

In [4]:
# 讀檔
assessments = pd.read_csv('assessments.csv')
student_assessments = pd.read_csv('studentAssessment.csv')
student_info = pd.read_csv('studentInfo.csv')

# 篩出非考試類型的作業
assignments_only = assessments[assessments['assessment_type'] != 'Exam']

# 合併實際繳交資料（左邊保留所有應繳交作業）
full_data = pd.merge(
    assignments_only,
    student_assessments,
    on='id_assessment',
    how='left',
    suffixes=('_assessment', '_student')
)

# 計算遲交欄位：僅當有繳交時再判斷
full_data['is_late'] = (full_data['date_submitted'] > full_data['date']) & full_data['date_submitted'].notna()

# 補上沒繳交的學生：用 assessments 去對 studentAssessment 的學生和作業做 Cartesian join
student_courses = student_info[['id_student', 'code_module', 'code_presentation']].drop_duplicates()

# 3. 將學生的課程修課紀錄與非考試作業 join，限制只對應到自己修的課程作業
assignment_students = pd.merge(
    assignments_only,
    student_courses,
    on=['code_module', 'code_presentation'],
    how='inner'  # 只對到有修課的 assignment
)

# 合併應繳資料與實際繳交情況（左邊是所有應該出現的 student-assignment 組合）
complete = pd.merge(
    assignment_students,
    full_data[['id_assessment', 'id_student', 'score', 'is_late']],
    on=['id_assessment', 'id_student'],
    how='left'
)

# 最終彙總：每位學生在每門課的資料
ass_stats = complete.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    total_submissions=('score', lambda x: x.notna().sum()),
    total_late=('is_late', lambda x: x.fillna(False).sum()),
    avg_score=('score', 'mean'),
    expected_assignments=('id_assessment', 'count')
).reset_index()

# 補上完成比例
ass_stats['late_ratio'] = ass_stats['total_late'] / ass_stats['total_submissions']
ass_stats['completion_ratio'] = ass_stats['total_submissions'] / ass_stats['expected_assignments']
ass_stats.drop(columns=['expected_assignments', 'total_late', 'total_submissions'], inplace=True)

# 顯示
ass_stats.head()

/var/folders/_q/xfvw_6_d3c5b8t1lpm7rppnh0000gn/T/ipykernel_25211/102821336.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  total_late=('is_late', lambda x: x.fillna(False).sum()),


,id_student,code_module,code_presentation,avg_score,late_ratio,completion_ratio
0,3733,DDD,2013J,NaN,NaN,0.000000
1,6516,AAA,2014J,61.800000,0.000000,1.000000
2,8462,DDD,2013J,87.666667,0.333333,0.500000
3,8462,DDD,2014J,86.500000,0.000000,0.666667
4,11391,AAA,2013J,82.000000,0.000000,1.000000


In [5]:
merged_df = pd.merge(merged_df, ass_stats, on=['code_module', 'code_presentation', 'id_student'], how='left')
print(merged_df.shape)

(4959, 24)


# Feature Fusion on Click Stats

In [7]:
activity_to_category = {
    'homepage': 'NAVIGATION',
    'page': 'NAVIGATION',
    'subpage': 'NAVIGATION',
    'sharedsubpage': 'INPUT',
    'resource': 'INPUT',
    'oucontent': 'INPUT',
    'htmlactivity': 'INPUT',
    'url': 'INPUT',
    'glossary': 'INPUT',
    'folder': 'INPUT',
    'dataplus': 'INPUT',
    'ouwiki': 'INPUT',
    'dualpane': 'INPUT',
    'forumng': 'OUTPUT',
    'oucollaborate': 'OUTPUT',
    'ouelluminate': 'OUTPUT',
    'quiz': 'OUTPUT',
    'externalquiz': 'OUTPUT',
    'questionnaire': 'OUTPUT',
    'repeatactivity': 'OUTPUT',
}

category = ['NAVIGATION', 'INPUT', 'OUTPUT'],

## 不同學習行為類型的分佈

In [9]:
# 上傳必要的檔案
df_path = "./studentVle.csv"
vle_path = "./vle.csv"

# 讀取資料
df = pd.read_csv(df_path)
vle = pd.read_csv(vle_path)

# 合併 activity_type
df = pd.merge(df, vle[['id_site', 'activity_type']], on='id_site', how='left')

# 加入分類欄位
df['category'] = df['activity_type'].map(activity_to_category)

# 結果表格初始化
final_results = []

# 對四大類別 + 全部做統計
for cat in category:
    if cat == 'ALL':
        subset = df.copy()
    else:
        subset = df[df['category'] == cat]
    
    # 按日期合併點擊數
    df_sorted = subset.sort_values(by=['code_module', 'code_presentation', 'id_student', 'id_site', 'date'])
    df_merged = df_sorted.groupby(['code_module', 'code_presentation', 'id_student', 'date']).agg(
        sum_click=('sum_click', 'sum')
    ).reset_index()

    # 日期間隔計算
    df_merged['date_diff'] = df_merged.groupby(['code_module', 'code_presentation', 'id_student'])['date'].diff()

    # 間隔統計
    interval_stats = df_merged.groupby(['code_module', 'code_presentation', 'id_student']).agg(
        mean_click_interval=('date_diff', 'mean'),
        std_click_interval=('date_diff', 'std'),
        mean_clicks=('sum_click', 'mean')
    ).reset_index()

    # 建立完整日期範圍
    def create_full_date_range(group):
        full_range = pd.DataFrame({'date': range(group['date'].min(), group['date'].max() + 1)})
        return full_range.merge(group, on='date', how='left').fillna({'sum_click': 0})

    df_full = df_merged.groupby(['code_module', 'code_presentation', 'id_student']).apply(create_full_date_range).reset_index(drop=True)

    # 變異係數計算
    df_full['click_variance'] = df_full.groupby(['code_module', 'code_presentation', 'id_student'])['sum_click'].transform(
        lambda x: np.std(x) / np.mean(x) if np.mean(x) != 0 else 0
    )

    variance_stats = df_full.groupby(['code_module', 'code_presentation', 'id_student']).agg(
        click_variance=('click_variance', 'first')
    ).reset_index()

    # 合併所有統計結果
    summary = pd.merge(interval_stats, variance_stats, on=['code_module', 'code_presentation', 'id_student'])
    summary.columns = ['code_module', 'code_presentation', 'id_student'] + [f'{cat.lower()}_{col}' for col in summary.columns[3:]]

    final_results.append(summary)

# 合併所有類別的統計結果
from functools import reduce
click_stats = reduce(lambda left, right: pd.merge(left, right, on=['code_module', 'code_presentation', 'id_student'], how='outer'), final_results)


/var/folders/_q/xfvw_6_d3c5b8t1lpm7rppnh0000gn/T/ipykernel_25211/130814166.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_full = df_merged.groupby(['code_module', 'code_presentation', 'id_student']).apply(create_full_date_range).reset_index(drop=True)
/var/folders/_q/xfvw_6_d3c5b8t1lpm7rppnh0000gn/T/ipykernel_25211/130814166.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_full = df_merged.g

In [10]:
click_stats.columns

Index(['code_module', 'code_presentation', 'id_student',
       'navigation_mean_click_interval', 'navigation_std_click_interval',
       'navigation_mean_clicks', 'navigation_click_variance',
       'processing_mean_click_interval', 'processing_std_click_interval',
       'processing_mean_clicks', 'processing_click_variance'],
      dtype='object')

In [11]:
merged_df = pd.merge(merged_df, click_stats, on=['code_module', 'code_presentation', 'id_student'], how='left')
print(merged_df.shape)

(4959, 32)


# Mean score of different learning phases

In [16]:
assessments = pd.read_csv('assessments.csv')
assessments = assessments[assessments['assessment_type'] != 'Exam']
student_assessments = pd.read_csv('studentAssessment.csv')
df = pd.merge(student_assessments, assessments, on='id_assessment')
course = pd.read_csv('courses.csv')

def label_stage(row, course_length):
    if row['date'] <= course_length * 0.33:
        return 'early'
    elif row['date'] <= course_length * 0.66:
        return 'mid'
    else:
        return 'late'

# 合併回學生資料並標記階段
df = pd.merge(df, course, on=['code_module', 'code_presentation'], how='left')
df['stage'] = df.apply(lambda row: label_stage(row, row['module_presentation_length']), axis=1)

if (weighted):
    # 定義一個計算加權平均的函數
    def weighted_avg(group):
        return (group['score'] * group['weight']).sum() / group['weight'].sum()

    # 依據 id_student, code_module, code_presentation, stage 分組，計算加權平均
    weighted_stage_scores = (
        df.groupby(['id_student', 'code_module', 'code_presentation', 'stage'])
        .apply(weighted_avg)
        .reset_index(name='weighted_score')
    )

    # 把 stage 欄位轉成欄位（pivot）
    stage_scores = weighted_stage_scores.pivot_table(
        index=['id_student', 'code_module', 'code_presentation'],
        columns='stage',
        values='weighted_score'
    ).reset_index()

else:
    # 計算每個階段的平均成績
    stage_scores = df.pivot_table(index=['id_student', 'code_module', 'code_presentation'],
                                columns='stage', values='score', aggfunc='mean').reset_index()
    stage_scores.rename(columns={'early': 'early_score', 'mid': 'mid_score', 'late': 'late_score'}, inplace=True)

In [18]:
merged_df = pd.merge(merged_df, stage_scores, on=['id_student', 'code_module', 'code_presentation'], how='left')
print(merged_df.shape)

(4959, 37)


# Export

In [19]:
# df = normalize(merged_df)
df = merged_df.copy()
if class_num == 3:
    df['score'] = pd.qcut(df['score'], 3, labels=['group1', 'group2', 'group3'])
    # Remove middle class
    df = df[df['score'] != 'group2']    
elif class_num == 5:
    df['score'] = pd.qcut(df['score'], 5, labels=['group1', 'group2', 'group3', 'group4', 'group5'])

# merge module and presentation
df['module_presentation'] = df['code_module'] + '_' + df['code_presentation']

df.drop(columns=['code_module','code_presentation', 'id_assessment','assessment_type','date','weight','id_student','date_submitted','is_banked','date_unregistration', 'num_of_prev_attempts','disability', 'final_result', 'avg_score'], inplace=True) # 'date_registration'
# std click interval is not used in final thesis
df.drop(columns=['navigation_std_click_interval', 'input_std_click_interval','output_std_click_interval'], inplace=True) 
df.dropna(inplace=True)

base_df = df[['score', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'module_presentation']]

base_df.to_csv(base_data_file_name, index=False)
df.to_csv(data_file_name, index=False)

In [23]:
# Export normalized data for Mixed Effect Model
normalized_df = normalize(df)
normalized_df.to_csv('data_normalized.csv', index=False)
normalized_base_df = normalized_df[['score', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'module_presentation']]
normalized_base_df.to_csv('data_base_normalized.csv', index=False)
normalized_df.describe()

       studied_credits  date_registration   late_ratio  completion_ratio  \
count      2811.000000        2811.000000  2811.000000       2811.000000   
mean          0.184198           0.661967     0.419017          0.930797   
std           0.138997           0.142342     0.325287          0.148602   
min           0.000000           0.000000     0.000000          0.375000   
25%           0.120000           0.576602     0.000000          1.000000   
50%           0.120000           0.701950     0.500000          1.000000   
75%           0.240000           0.779944     0.625000          1.000000   
max           1.000000           1.000000     1.000000          1.000000   

       navigation_mean_click_interval  navigation_std_click_interval  \
count                     2811.000000                    2811.000000   
mean                         0.138564                       0.167511   
std                          0.113022                       0.114015   
min                        